# Machine Learning Project in Azure Databricks

## Heart Disease Prediction and Student Performance Prediction

This notebook demonstrates a machine learning workflow using Azure Databricks, PySpark, and Spark MLlib.

In [0]:
heart_df = spark.table("gr21_databricks_7405608240565156.default.heart_dataset")
student_df = spark.table("gr21_databricks_7405608240565156.default.student_dataset")

display(heart_df)
display(student_df)

age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
57,0,0,120,354,0,1,163,1,0.6,2,0,2,1
57,1,0,140,192,0,1,148,0,0.4,1,0,1,1
56,0,1,140,294,0,0,153,0,1.3,1,0,2,1
44,1,1,120,263,0,1,173,0,0.0,2,0,3,1
52,1,2,172,199,1,1,162,0,0.5,2,0,3,1
57,1,2,150,168,0,1,174,0,1.6,2,0,2,1


school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10
GP,M,16,U,LE3,T,4,3,services,other,reputation,mother,1,2,0,no,yes,yes,yes,yes,yes,yes,no,5,4,2,1,2,5,10,15,15,15
GP,M,16,U,LE3,T,2,2,other,other,home,mother,1,2,0,no,no,no,no,yes,yes,yes,no,4,4,4,1,1,3,0,12,12,11
GP,F,17,U,GT3,A,4,4,other,teacher,home,mother,2,2,0,yes,yes,no,no,yes,yes,no,no,4,1,4,1,1,1,6,6,5,6
GP,M,15,U,LE3,A,3,2,services,other,home,mother,1,2,0,no,yes,yes,no,yes,yes,yes,no,4,2,2,1,1,1,0,16,18,19
GP,M,15,U,GT3,T,3,4,other,other,home,mother,1,2,0,no,yes,yes,yes,yes,yes,yes,no,5,5,1,1,1,5,0,14,15,15


In [0]:
print("Heart dataset rows:", heart_df.count())
print("Heart dataset columns:", heart_df.columns)

print("Student dataset rows:", student_df.count())
print("Student dataset columns:", student_df.columns)

Heart dataset rows: 303
Heart dataset columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
Student dataset rows: 395
Student dataset columns: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


In [0]:
from pyspark.sql import functions as F

print("Null values in Heart Disease dataset:")

heart_nulls = heart_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in heart_df.columns
])

heart_nulls.show()

print("Null values in Student Performance dataset:")

student_nulls = student_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in student_df.columns
])

student_nulls.show()

Null values in Heart Disease dataset:
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
|age|sex| cp|trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
|  0|  0|  0|       0|   0|  0|      0|      0|    0|      0|    0|  0|   0|     0|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+

Null values in Student Performance dataset:
+------+---+---+-------+-------+-------+----+----+----+----+------+--------+----------+---------+--------+---------+------+----+----------+-------+------+--------+--------+------+--------+-----+----+----+------+--------+---+---+---+
|school|sex|age|address|famsize|Pstatus|Medu|Fedu|Mjob|Fjob|reason|guardian|traveltime|studytime|failures|schoolsup|famsup|paid|activities|nursery|higher|internet|romantic|famrel|freetime|goout|Dalc|Walc|health|absences| G1| G2| G3|
+------+---+---+-------+-------

## Heart Disease Prediction Model

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col

heart_clean = heart_df.dropna()

target_column = "target"
feature_columns = [c for c in heart_clean.columns if c != target_column]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

heart_data = assembler.transform(heart_clean).select(
    "features",
    col(target_column).alias("label")
)

train_heart, test_heart = heart_data.randomSplit([0.8, 0.2], seed=42)

heart_model = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50
)

trained_heart_model = heart_model.fit(train_heart)

heart_predictions = trained_heart_model.transform(test_heart)

display(heart_predictions.select("label", "prediction", "probability"))

label,prediction,probability
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.10677691833770468"",""0.8932230816622954""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.08928271656216938"",""0.9107172834378306""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.12840235854458698"",""0.8715976414554131""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2567712277319294"",""0.7432287722680706""]}"
0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.13000118973123456"",""0.8699988102687654""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2580104148255185"",""0.7419895851744814""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.1573949100184489"",""0.8426050899815511""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0881637911225912"",""0.9118362088774087""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.22675534225449476"",""0.7732446577455053""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.038359932862442485"",""0.9616400671375576""]}"


In [0]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

heart_accuracy = accuracy_evaluator.evaluate(heart_predictions)
heart_f1 = f1_evaluator.evaluate(heart_predictions)

print("Heart Disease Prediction Accuracy:", heart_accuracy)
print("Heart Disease Prediction F1 Score:", heart_f1)

Heart Disease Prediction Accuracy: 0.8085106382978723
Heart Disease Prediction F1 Score: 0.8112345268000363


## Student Performance Prediction Model

In [0]:
import gc

try:
    del trained_student_model
except:
    pass

try:
    del student_predictions
except:
    pass

try:
    del student_pipeline
except:
    pass

gc.collect()

16061

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import when, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Lexo tabelën
student_df = spark.table("gr21_databricks_7405608240565156.default.student_dataset")

# Pastro null values
student_clean = student_df.dropna()

# Krijo label: 1 = pass, 0 = fail
student_clean = student_clean.withColumn(
    "label",
    when(col("G3").cast("double") >= 10, 1.0).otherwise(0.0)
)

# Përdor vetëm kolona numerike, pa kategori
numeric_columns = [
    "age",
    "Medu",
    "Fedu",
    "traveltime",
    "studytime",
    "failures",
    "famrel",
    "freetime",
    "goout",
    "Dalc",
    "Walc",
    "health",
    "absences"
]

# Cast në double
for c in numeric_columns:
    student_clean = student_clean.withColumn(c, col(c).cast("double"))

# VectorAssembler
assembler_student = VectorAssembler(
    inputCols=numeric_columns,
    outputCol="features",
    handleInvalid="keep"
)

student_features = assembler_student.transform(student_clean)

# Merr vetëm features dhe label
student_model_data = student_features.select("features", "label")

# Train/Test split
train_student, test_student = student_model_data.randomSplit([0.8, 0.2], seed=42)

# Random Forest më i lehtë
student_rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=3,
    maxDepth=4,
    seed=42
)

# Train model
student_rf_model = student_rf.fit(train_student)

# Predictions
student_predictions = student_rf_model.transform(test_student)

# Shfaq rezultatet
display(
    student_predictions.select(
        "label",
        "prediction",
        "probability"
    )
)

# Accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(student_predictions)

print("Student Performance Accuracy:", accuracy)

label,prediction,probability
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8349161126938904"",""0.16508388730610954""]}"
1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.29759801824805204"",""0.7024019817519479""]}"
1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2904411028785763"",""0.7095588971214237""]}"
1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2904411028785763"",""0.7095588971214237""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2453159381881942"",""0.7546840618118059""]}"
1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.14986139273364874"",""0.8501386072663513""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6538461538461539"",""0.3461538461538462""]}"
1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6538461538461539"",""0.3461538461538462""]}"
1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.38106995884773665"",""0.6189300411522635""]}"
1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.14077048364273967"",""0.8592295163572604""]}"


Student Performance Accuracy: 0.7543859649122807


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = accuracy_evaluator.evaluate(student_predictions)
precision = precision_evaluator.evaluate(student_predictions)
recall = recall_evaluator.evaluate(student_predictions)
f1 = f1_evaluator.evaluate(student_predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.7543859649122807
Precision: 0.7413494209832883
Recall: 0.7543859649122806
F1 Score: 0.7468810916179338


## Conclusion

In this project, two classification problems were implemented in Azure Databricks.

The first model predicts whether a patient has heart disease based on medical attributes.  
The second model predicts whether a student passes or fails based on demographic and academic features.

The datasets were uploaded as Databricks tables, loaded using Spark, processed with PySpark, and trained using Random Forest classification models. The models were evaluated using Accuracy and F1 Score.